# GEAR Ultimate — Metal-binding residue prediction

**One notebook / modular Python backend / resumable Drive outputs**

This notebook consolidates the full experiment lineage into a single reproducible workflow:

1. **Historical base reproduction** — original ESM + DSSP/dihedrals + KAN, including the historical label-aware spatial smoothness term.
2. **Standardized 13-representation benchmark** — MLP, KAN, and XGBoost under matched official 5-fold protein splits.
3. **Mechanism-first extensions** — ATOM, CHEM, MECH1, MULTI2 and their ablations.
4. **Final focused ESM comparison** — the six representations used in the final external comparison.
5. **No-ESM controls** — compact explicit chemistry/geometry and SEQCTX controls.
6. **Graph experiments** — maximal graph, clean pure graph, residual graph, donor-focused graph.
7. **Evolutionary extension** — ColabFold/MMseqs2 MSA → EVO48; Dev-first screening, then external confirmation.
8. **Reporting / leakage audit / error-overlap analysis**.

The notebook itself stays deliberately small. All scientific implementation lives in `gear_ultimate/*.py`, so feature definitions and training logic are not duplicated across cells.

---

## Experimental contract

- **Task:** residue-level `metal_binding` prediction.
- **Development:** bindNode24 official protein-level 5-fold split on DevSet1010.
- **External confirmation:** TestSet300 and TestSetNew46.
- **Primary metric:** AUPRC. MCC and F1 are secondary; AUROC is contextual.
- **Threshold:** select once from pooled Dev OOF predictions; freeze before external evaluation.
- **External sets are confirmation sets, not tuning sets.** Repeated historical inspection means a genuinely final claim should ideally use a new blind external cohort.
- **No-ESM ≠ fully PLM-free:** structures are ESMFold-derived unless replaced with experimental/non-PLM structures.

### Important historical nuance

The original notebook called the model “no graph”, because no GNN message passing occurs in the forward pass. However, its training loss includes **label-aware spatial smoothness over Cα edges**. The historical KAN run is therefore preserved separately and must not be described as a strictly structure-free baseline.


## 0. Drive layout

Upload/extract the complete `GEAR_ULTIMATE` folder to:

```text
/content/drive/MyDrive/Protein-binding-Phi/GEAR_ULTIMATE
```

The existing project is expected to contain:

```text
/content/drive/MyDrive/Protein-binding-Phi/
├── data/development_set/full_grouped_train_binding_sites_df.csv
├── data/development_set/full_grouped_test_binding_sites_df.csv
├── data/independent_set/grouped_test_46_new_binding_sites.csv
└── esmFold_pdb_files/
```

Existing compatible caches are reused. Every new experiment gets its own `run_version`, so old standardized results are not overwritten.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')
ULTIMATE_DIR = BASE / 'GEAR_ULTIMATE'

if not ULTIMATE_DIR.exists():
    local = Path('/content/GEAR_ULTIMATE')
    if local.exists():
        ULTIMATE_DIR = local
    else:
        raise FileNotFoundError(
            'GEAR_ULTIMATE folder not found. Upload/extract the supplied bundle to '\
            '/content/drive/MyDrive/Protein-binding-Phi/GEAR_ULTIMATE first.'
        )

print('BASE:', BASE)
print('ULTIMATE_DIR:', ULTIMATE_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: /content/drive/MyDrive/Protein-binding-Phi
ULTIMATE_DIR: /content/drive/MyDrive/Protein-binding-Phi/GEAR_ULTIMATE


In [ ]:
from pathlib import Path
import os

BASE = Path("/content/drive/MyDrive/Protein-binding-Phi")

print("=== GPU ===")
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print("\n=== EXISTING FEATURE CACHE ===")

CACHE = BASE / "cache"

if CACHE.exists():
    files = [p for p in CACHE.rglob("*") if p.is_file()]
    print("Cache files:", len(files))
    print(
        "Total cache:",
        round(sum(p.stat().st_size for p in files)/1024**3, 2),
        "GB"
    )

    keywords = [
        "esm",
        "spatial",
        "local",
        "atom",
        "chem",
        "access",
        "shell",
        "multi",
        "seqctx",
    ]

    for key in keywords:
        hits = [
            p for p in files
            if key.lower() in str(p).lower()
        ]
        size = sum(p.stat().st_size for p in hits)
        print(
            f"{key:10s}: "
            f"{len(hits):5d} files | "
            f"{size/1024**3:.3f} GB"
        )

print("\n=== NEW V2 EXPERIMENT CACHE ===")

versions = [
    "metal_v2_01_family_mlp",
    "metal_v2_02_component_mlp",
    "metal_v2_03_combo_mlp",
    "metal_v2_04_locked_model_family",
    "metal_v2_05_graph_diagnostic",
    "metal_v2_06_final_locked_external",
]

for v in versions:
    d = BASE / "results" / v / "metal_binding"

    if not d.exists():
        print(f"{v:42s}: NOT STARTED")
        continue

    fs = [p for p in d.rglob("*") if p.is_file()]

    print(
        f"{v:42s}: "
        f"{len(fs)} files | "
        f"{sum(p.stat().st_size for p in fs)/1024**2:.1f} MB"
    )

=== GPU ===
CUDA available: True
GPU: NVIDIA L4
VRAM: 22.03 GB

=== EXISTING FEATURE CACHE ===
Cache files: 12437
Total cache: 8.05 GB
esm       :  1359 files | 0.747 GB
spatial   :    21 files | 0.825 GB
local     :     4 files | 0.126 GB
atom      :     3 files | 0.013 GB
chem      :     3 files | 0.021 GB
access    :     0 files | 0.000 GB
shell     :     0 files | 0.000 GB
multi     :     3 files | 0.052 GB
seqctx    :     0 files | 0.000 GB

=== NEW V2 EXPERIMENT CACHE ===
metal_v2_01_family_mlp                    : NOT STARTED
metal_v2_02_component_mlp                 : NOT STARTED
metal_v2_03_combo_mlp                     : NOT STARTED
metal_v2_04_locked_model_family           : NOT STARTED
metal_v2_05_graph_diagnostic              : NOT STARTED
metal_v2_06_final_locked_external         : NOT STARTED


In [ ]:
# %%bash
# BASE="/content/drive/MyDrive/Protein-binding-Phi"
# TMP="/content/_unzip_tmp"
# rm -rf "$TMP" && mkdir -p "$TMP"

# for f in "$BASE"/Protein-binding-20260830T031044Z-1-00{1,2}.zip; do
#   unzip -q -o "$f" -d "$TMP"
# done

# # nếu bên trong chỉ có 1 folder gốc thì lấy ruột của nó, không thì lấy hết
# inner=$(find "$TMP" -mindepth 1 -maxdepth 1)
# if [ "$(echo "$inner" | wc -l)" -eq 1 ] && [ -d "$inner" ]; then
#   src="$inner"
# else
#   src="$TMP"
# fi

# shopt -s dotglob
# cp -r "$src"/* "$BASE"/
# rm -rf "$TMP"
# ls -la "$BASE"

## 1. Install dependencies

`pykan` is included because the **standardized modern suite** can train KAN alongside MLP and XGBoost. The historical KAN protocol is also available separately.

The evolutionary branch has heavier ColabFold/AlphaFold dependencies, so it is installed only when requested.


In [ ]:
# Core dependencies. Re-running is safe.
import subprocess
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(ULTIMATE_DIR / 'requirements.txt')
])


0

In [ ]:
# Optional: only enable when you are actually generating MSAs.
INSTALL_EVOLUTION_DEPS = False

if INSTALL_EVOLUTION_DEPS:
    import subprocess
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '-U',
        'colabfold[alphafold]'
    ])


## 2. Import the modular backend

### Module map

| Module | Responsibility |
|---|---|
| `gear_core.py` | standardized data, ATOM/CHEM, MLP/KAN/XGB, metrics, OOF thresholding, external ensemble |
| `gear_structure.py` | ESM-2 1280-d, SPATIAL112, LOCAL8 caches |
| `gear_mechanistic.py` | MECH1, MULTI2, SEQCTX182 |
| `gear_legacy.py` | historical original KAN protocol |
| `gear_graph.py` | 18-edge relation graph, maximal/clean/residual/donor-focused experiments |
| `gear_evolution.py` | batch/retry/resume MSA + EVO48 |
| `gear_suite.py` | experiment grids and orchestration |
| `gear_reporting.py` | final comparison and prediction-overlap helpers |

The graph backend is a faithful reconstruction of the frozen graph specification used in the later experiments. Because the original clean-graph one-cell source was not among the recovered source files, graph runs use **new run versions** and should not be expected to produce byte-identical old checkpoints. The historical/tabular feature code was consolidated directly from recovered notebooks/scripts.


In [ ]:
sys.path.insert(0, str(ULTIMATE_DIR))

import numpy as np
import pandas as pd
import torch

from gear_ultimate import __version__ as GEAR_ULTIMATE_VERSION
from gear_ultimate import gear_core as core
from gear_ultimate import gear_structure as structure
from gear_ultimate import gear_mechanistic as mech
from gear_ultimate import gear_legacy as legacy
from gear_ultimate import gear_graph as graph
from gear_ultimate import gear_evolution as evo
from gear_ultimate import gear_suite as suite
from gear_ultimate import gear_reporting as reporting

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('GEAR Ultimate:', GEAR_ULTIMATE_VERSION)
print('DEVICE:', DEVICE)
print('Torch:', torch.__version__)
print('KAN available:', core.HAS_KAN)


GEAR Ultimate: 1.0.0
DEVICE: cuda
Torch: 2.11.0+cu128
KAN available: True


## 3. Master switches

Nothing expensive runs unless its switch is `True`. This lets the notebook act as both a complete archive and a practical day-to-day runner.

Recommended use:

- On a fresh project, run `PREPARE_CONTEXT=True` once; caches then resume/reuse.
- For a paper-quality matched benchmark, run the standardized suite with all three model families.
- For feature discovery, run MLP first.
- For EVO, **finish Dev MSA first** and run Dev-only CV before spending time on external MSAs.


In [ ]:
# --------------------------
# Data / feature preparation
# --------------------------
PREPARE_CONTEXT = True
ESM_BATCH_SIZE = 8

# --------------------------
# Historical reproduction
# --------------------------
RUN_HISTORICAL_KAN = False

# --------------------------
# Standardized experiments
# --------------------------
RUN_STANDARDIZED_13 = False
STANDARDIZED_MODELS = ('MLP', 'KAN', 'XGBoost')

RUN_MECHANISTIC_GRID = False
MECHANISTIC_MODELS = ('MLP',)

RUN_FINAL_FOCUSED = False
FINAL_MODELS = ('MLP',)

RUN_NO_ESM_GRID = False
NO_ESM_MODELS = ('MLP',)

# --------------------------
# Graph diagnostics
# --------------------------
RUN_MAXIMAL_GRAPH = False
RUN_CLEAN_GRAPH = False

# --------------------------
# Evolutionary branch
# --------------------------
RUN_MSA_DEV = False
MSA_BATCH_SIZE = 25
MSA_MAX_BATCHES = None   # e.g. 5 for a short resumable chunk, None for all missing Dev

ATTACH_EVO_DEV = False
RUN_EVO_DEV_CV = False
EVO_DEV_MODELS = ('MLP',)

RUN_MSA_EXTERNAL = False
ATTACH_EVO_ALL = False
RUN_EVO_FULL = False
EVO_FULL_MODELS = ('MLP',)

RUN_EVO_GRAPH = False

print('Switches loaded.')


Switches loaded.


## 4. Build the unified data context

This stage creates/reuses all label-independent representations needed by the non-EVO experiments:

- **AA 20**
- **ESM 1280**: ESM-2 `esm2_t33_650M_UR50D`, mean of the last two hidden layers
- **SPATIAL 112**
- **LOCAL 8**
- **ATOM 15**
- **CHEM 24**
- **MECH1 34** = ACCESS10 + SHELL2 12 + ENV8 + CONF4
- **MULTI2 58** = METALSPEC18 + ELEC3D8 + TOPO12 + SOLV10 + SEQMOTIF10
- **SEQCTX 182**

All structure-derived feature caches are computed from the ESMFold PDBs already used throughout the project.


In [ ]:
ctx = None

if PREPARE_CONTEXT:
    ctx = suite.prepare_context(
        base=BASE,
        run_version='ultimate_context_v1',
        models=('MLP','KAN','XGBoost'),
        ensure_base=True,
        add_mech=True,
        add_multi=True,
        add_seqctx=True,
        esm_batch_size=ESM_BATCH_SIZE,
        device=DEVICE,
    )
    display(suite.audit_context(ctx))
    display(reporting.dimension_table(ctx.stores['train']))


[DevSet1010] Reused ESM cache: DevSet1010_ESM2_t33_mean_last2_1280_fp16.pkl
[DevSet1010] Reused structure cache: DevSet1010_spatial_local_v2.pkl
[TestSet300] Reused ESM cache: TestSet300_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSet300] Reused structure cache: TestSet300_spatial_local_v2.pkl
[TestSetNew46] Reused ESM cache: TestSetNew46_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSetNew46] Reused structure cache: TestSetNew46_spatial_local_v2.pkl
[DevSet1010] Reused ATOM cache: DevSet1010_atomic_coordination_v1.pkl
[DevSet1010] Reused CHEM cache: DevSet1010_chemistry_v1_24d.pkl
[TestSet300] Reused ATOM cache: TestSet300_atomic_coordination_v1.pkl
[TestSet300] Reused CHEM cache: TestSet300_chemistry_v1_24d.pkl
[TestSetNew46] Reused ATOM cache: TestSetNew46_atomic_coordination_v1.pkl
[TestSetNew46] Reused CHEM cache: TestSetNew46_chemistry_v1_24d.pkl
[DevSet1010] Reused MECH1 cache: DevSet1010_mechanistic_v1.pkl
[DevSet1010] Reused MULTI2 cache: DevSet1010_multisignal_v2.pkl
[TestSet300] Reused 

,split,proteins,residues,positives,prevalence
0,train,1010,170181,2370,0.013926
1,test300,300,62689,881,0.014054
2,test46,46,6227,77,0.012366


,block,dim
0,AA,20
6,ACCESS,10
4,ATOM,15
5,CHEM,24
9,CONF,4
11,ELEC3D,8
8,ENV,8
1,ESM,1280
3,LOCAL,8
10,METALSPEC,18


## 5. Reproducibility / leakage audit

The modern standardized branch follows these rules:

1. Protein-level official folds; no residue-level random split.
2. Scalers are fit on **training-fold residues only**.
3. Positive weighting/class imbalance parameters are estimated from the training fold.
4. Feature extraction does not read the binding labels.
5. Model selection is based on Dev validation AUPRC.
6. OOF threshold is chosen from Dev OOF predictions only.
7. External predictions are the mean of the five fold-specific models.
8. Test300/Test46 should not drive architecture or feature choices.

### Known limitation

Early stopping and reported validation performance use the same fold, so there is ordinary model-selection optimism. AUPRC itself is threshold-independent, but it is still measured on the fold used for early stopping. The external sets have also been inspected repeatedly across this research sequence, so they are best treated as **confirmation**, not pristine blind tests.


In [ ]:
if ctx is not None:
    explicit_blocks = [
        'AA','SPATIAL','LOCAL','ATOM','CHEM','ACCESS','SHELL2','ENV','CONF',
        'METALSPEC','ELEC3D','TOPO','SOLV','SEQMOTIF','SEQCTX'
    ]
    display(reporting.validate_no_direct_label_features(ctx.stores, explicit_blocks))


,split,block,missing_proteins,mean_finite_fraction
0,train,AA,0,1.0
1,train,SPATIAL,0,1.0
2,train,LOCAL,0,1.0
3,train,ATOM,0,1.0
4,train,CHEM,0,1.0
5,train,ACCESS,0,1.0
6,train,SHELL2,0,1.0
7,train,ENV,0,1.0
8,train,CONF,0,1.0
9,train,METALSPEC,0,1.0


# Part A — Historical original KAN baseline

This is intentionally **not merged into the standardized protocol**, because it answers a different reproducibility question: *can we reproduce the behavior of the original notebook?*

Historical architecture/protocol:

- ESM-2 residue embedding: 1280 d.
- DSSP H/E/C: 3 d.
- raw φ/ψ: 2 d.
- total node vector: 1285 d.
- `Linear(1285, 32)` → KAN `[32, 8, 2]`, grid=2, k=2.
- weighted focal-style cross entropy.
- label-aware Cα smoothness edges: cutoff 17 Å, max 64 neighbors.
- positive weight 9, γ=2, position weight 1.5.
- seed 15, lr 1e-4, weight decay 0.015, 50 epochs, patience 10.
- early stopping on validation F1 at threshold 0.5.

This historical result is useful as an archival baseline, but **the standardized suite below is the fair comparison platform**.


In [ ]:
if RUN_HISTORICAL_KAN:
    if not core.HAS_KAN:
        raise RuntimeError('KAN is not available. Re-run the dependency cell and verify pykan.')

    hist_cfg = legacy.HistoricalConfig(base=str(BASE))
    historical_result = legacy.run_historical_baseline(hist_cfg, device=DEVICE)
    historical_result


# Part B — Standardized 13-representation benchmark

The standardized grid keeps the split, scaling, early-stopping metric and reporting framework matched across **MLP / KAN / XGBoost**.

Representations:

```text
AA
AA+Structure
AA+Atomic
AA+Chem
AA+Atomic+Chem
ESM
ESM+Structure
ESM+Atomic
ESM+Chem
ESM+Structure+Atomic
ESM+Structure+Chem
ESM+Atomic+Chem
ESM+Structure+Atomic+Chem
```

This stage reproduces the core observation that explicit local chemistry/coordination is powerful, but much of generic coarse structure is redundant once ESM is present.


In [ ]:
if RUN_STANDARDIZED_13:
    std_ctx = suite.clone_context_for_run(
        ctx,
        run_version='ultimate_standardized_13_v1',
        models=STANDARDIZED_MODELS,
    )
    std_cv, std_ext, _ = suite.run_grid(
        std_ctx,
        suite.STANDARDIZED_13,
        models=STANDARDIZED_MODELS,
    )
    display(std_cv)
    display(std_ext.sort_values(['dataset','model','AUPRC'], ascending=[True,True,False]))


# Part C — Mechanism-first feature families

## ATOM15

Explicit protein-side coordination geometry around canonical donor atoms (Cys SG, His ND1/NE2, Asp OD1/OD2, Glu OE1/OE2), including donor counts/distances, candidate-center fit, clearance and angular organization.

## CHEM24

Local donor-type chemistry and charge/electrostatic proxies, including carboxylate O, imidazole N, thiol S, hydroxyl O, amide O, thioether S and backbone carbonyl O counts.

## MECH1 = 34 d

- ACCESS 10 — packing/open-space/accessibility proxies
- SHELL2 12 — second-shell support
- ENV 8 — local physicochemical environment
- CONF 4 — ESMFold confidence context

## MULTI2 = 58 d

- METALSPEC 18 — Zn/Mg/Ca/Mn/Fe/Cu heuristic compatibility
- ELEC3D 8 — local 3D electrostatic field proxies
- TOPO 12 — donor-network topology/cooperativity
- SOLV 10 — solvation/polar organization
- SEQMOTIF 10 — explicit donor/acidic sequence motifs

The current evidence favored MULTI2 as a useful *combination*, with topology the strongest single new family, but none of the multiple exploratory feature additions survived Holm correction individually.


In [ ]:
if RUN_MECHANISTIC_GRID:
    mech_ctx = suite.clone_context_for_run(
        ctx,
        run_version='ultimate_mechanistic_grid_v1',
        models=MECHANISTIC_MODELS,
    )
    mech_cv, mech_ext, mech_paired = suite.run_grid(
        mech_ctx,
        suite.MECHANISTIC_GRID,
        models=MECHANISTIC_MODELS,
        comparisons=suite.MECHANISTIC_COMPARISONS,
    )
    display(mech_cv)
    display(mech_paired.sort_values(['model','mean_delta'], ascending=[True,False]))
    display(mech_ext.sort_values(['dataset','model','AUPRC'], ascending=[True,True,False]))


# Part D — Final focused ESM representation comparison

This is the compact grid used to answer the main representation question without rerunning every exploratory family:

1. ESM
2. ESM + Atomic
3. ESM + Chem
4. ESM + Atomic + Chem
5. ESM + MULTI2
6. **ESM + Atomic + Chem + MULTI2**

The historically observed final model had the highest AUPRC on both external sets. That does **not** mean every secondary metric was best.


In [ ]:
if RUN_FINAL_FOCUSED:
    final_ctx = suite.clone_context_for_run(
        ctx,
        run_version='ultimate_final_focused_v1',
        models=FINAL_MODELS,
    )
    final_cv, final_ext, _ = suite.run_grid(
        final_ctx,
        suite.FINAL_COMPARISON_REPS,
        models=FINAL_MODELS,
    )
    display(final_cv)
    display(final_ext.sort_values(['dataset','model','AUPRC'], ascending=[True,True,False]))


# Part E — No-ESM compact controls

The no-ESM branch tests whether explicit biology can replace the residue language-model embedding.

Key controls:

- AA + Atomic + Chem
- AA + Atomic + Chem + MULTI2
- SEQCTX + Atomic + Chem + MULTI2
- AA + SEQCTX + Atomic + Chem + MULTI2
- CompactFull = AA + SEQCTX + Atomic + Chem + MULTI2 + ACCESS + SHELL2
- ESM + Atomic + Chem + MULTI2 reference

`SEQCTX182` deliberately encodes simple local sequence information so that a failure cannot be blamed merely on omitting nearby amino-acid identities. Its previous performance suggested that the remaining ESM advantage is not just a ±4/±8 local motif effect.


In [ ]:
if RUN_NO_ESM_GRID:
    noesm_ctx = suite.clone_context_for_run(
        ctx,
        run_version='ultimate_no_esm_v1',
        models=NO_ESM_MODELS,
    )
    noesm_cv, noesm_ext, _ = suite.run_grid(
        noesm_ctx,
        suite.NO_ESM_GRID,
        models=NO_ESM_MODELS,
    )
    display(noesm_cv)
    display(noesm_ext.sort_values(['dataset','model','AUPRC'], ascending=[True,True,False]))


# Part F — Graph experiments

## Relation graph

Each protein is a residue graph with 18 edge descriptors covering:

- Cα distance / radial kernels
- sequence separation
- canonical donor identity
- donor–donor atom distance
- likely first-shell coordination relation
- donor↔polar second-shell support
- acidic/basic/opposite-charge relations

Two edge modes are available:

- `rich`: all accepted relation edges
- `donor`: chemistry-focused subset

## Why two graph stages?

### Maximal graph
The maximal diagnostic puts essentially every no-ESM feature family on each node. It is kept as an archival stress test. Historically it **underperformed** the compact tabular representation, supporting the idea that redundant local summaries plus message passing create noise/oversmoothing.

### Clean graph
The clean graph asks a sharper question: does an explicit relational inductive bias add information that compact independent-residue features miss?

- `PureGraph_AA_Chem`
- `PureGraph_AA_Atomic_Chem`
- `ResidualGraph_AA_Atomic_Chem`
- `ResidualGraph_Compact`
- `ResidualGraph_DonorFocused`

Residual graph prediction is:

\[
\text{logit}_i = \text{local-logit}_i + \sigma(\alpha)\,\Delta\text{graph}_i
\]

with α initialized small (~0.10). This matches the error-overlap observation that new structural mechanisms should correct a small hard-case subset rather than overwrite the local predictor.


In [ ]:
if RUN_MAXIMAL_GRAPH:
    max_cv, max_ext = graph.run_graph_grid(
        ctx.stores,
        ctx.fold_ids,
        BASE,
        graph.MAXIMAL_GRAPH_EXPERIMENT,
        run_version='ultimate_maximal_noesm_graph_v1',
        cfg=graph.maximal_graph_config(),
        device=DEVICE,
    )
    display(max_cv)
    display(max_ext)


In [ ]:
if RUN_CLEAN_GRAPH:
    clean_cv, clean_ext = graph.run_graph_grid(
        ctx.stores,
        ctx.fold_ids,
        BASE,
        graph.CLEAN_GRAPH_EXPERIMENTS,
        run_version='ultimate_clean_graph_v2',
        cfg=graph.GraphConfig(),
        device=DEVICE,
    )
    display(clean_cv)
    display(clean_ext.sort_values(['dataset','AUPRC'], ascending=[True,False]))


# Part G — Evolutionary information (EVO48)

This branch targets the most plausible information class still missing after extensive local chemistry/geometry engineering: **evolutionary conservation and homolog statistics**.

Per residue, EVO48 contains:

- 20 MSA amino-acid profile frequencies
- 20 profile log-odds
- normalized entropy
- gap/unknown fraction
- log homolog depth
- focal-AA frequency
- H/C/D/E donor-group frequency
- D/E acidic-group frequency
- H/C frequency
- homolog-availability flag

No binding label and no ESM residue embedding is used to compute EVO.

## Dev-first rule

The public ColabFold/MMseqs2 service is rate-limited. Therefore:

1. Generate **DevSet1010 MSAs first** in small resumable batches.
2. Run Dev-only EVO CV.
3. Only if EVO clearly improves Dev CV, spend time generating Test300/Test46 MSAs.

Do not parallelize many public-server workers.


In [ ]:
if RUN_MSA_DEV:
    evo.run_msa_batches(
        ctx.stores,
        BASE,
        splits=('train',),
        batch_size=MSA_BATCH_SIZE,
        max_retries=4,
        sleep_between=8,
        msa_mode='mmseqs2_uniref_env',
        max_batches=MSA_MAX_BATCHES,
    )


In [ ]:
if ATTACH_EVO_DEV:
    dev_evo_coverage = evo.attach_evo(
        ctx.stores,
        BASE,
        splits=('train',),
        require_msa=True,
    )
    display(dev_evo_coverage)


### Dev-only EVO test

The first scientific question is not “does EVO beat ESM?” but:

\[
\text{Compact} \rightarrow \text{Compact+EVO}: \quad \Delta\text{AUPRC} > 0\ ?
\]

The most informative comparisons are:

- AA + EVO
- AA + Chem + EVO
- AA + Atomic + Chem + EVO
- AA + Atomic + Chem + MULTI2 + EVO
- ESM + Atomic + Chem + MULTI2 reference

The reference is included only to quantify the remaining context gap; EVO feature selection should still be based on Dev CV.


In [ ]:
if RUN_EVO_DEV_CV:
    evo_dev_cv, evo_dev_folds = suite.run_dev_cv_only(
        ctx,
        suite.EVO_GRID,
        models=EVO_DEV_MODELS,
        run_version='ultimate_evo_dev_only_v1',
    )
    display(evo_dev_cv.sort_values(['model','CV_AUPRC'], ascending=[True,False]))


### External EVO confirmation — only after Dev signal

This stage fetches missing MSAs for Test300/Test46, attaches EVO to all splits, and then runs the full matched CV + external ensemble reporting.


In [ ]:
if RUN_MSA_EXTERNAL:
    evo.run_msa_batches(
        ctx.stores,
        BASE,
        splits=('test300','test46'),
        batch_size=MSA_BATCH_SIZE,
        max_retries=4,
        sleep_between=8,
        msa_mode='mmseqs2_uniref_env',
        max_batches=None,
    )

if ATTACH_EVO_ALL:
    all_evo_coverage = evo.attach_evo(
        ctx.stores,
        BASE,
        splits=('train','test300','test46'),
        require_msa=True,
    )
    display(all_evo_coverage)

if RUN_EVO_FULL:
    evo_ctx = suite.clone_context_for_run(
        ctx,
        run_version='ultimate_evo_full_v1',
        models=EVO_FULL_MODELS,
    )
    evo_cv, evo_ext, _ = suite.run_grid(
        evo_ctx,
        suite.EVO_GRID,
        models=EVO_FULL_MODELS,
    )
    display(evo_cv)
    display(evo_ext.sort_values(['dataset','model','AUPRC'], ascending=[True,True,False]))


## EVO + graph

Only run this after EVO exists for **all three splits**. It asks whether evolutionary node information and explicit relational structure are complementary.

The grid stays deliberately small; we do not return to a 453-dimensional maximal node representation.


In [ ]:
EVO_GRAPH_EXPERIMENTS = {
    'PureGraph_AA_Chem_EVO': {
        'blocks':['AA','CHEM','EVO'], 'architecture':'pure', 'edge_mode':'rich'
    },
    'ResidualGraph_AA_Atomic_Chem_EVO': {
        'blocks':['AA','ATOM','CHEM','EVO'], 'architecture':'residual', 'edge_mode':'rich'
    },
    'ResidualGraph_DonorFocused_EVO': {
        'blocks':['AA','ATOM','CHEM','EVO'], 'architecture':'residual', 'edge_mode':'donor'
    },
    'ResidualGraph_Compact_EVO': {
        'blocks':['AA','ATOM','CHEM',*suite.MULTI2,'EVO'], 'architecture':'residual', 'edge_mode':'rich'
    },
}

if RUN_EVO_GRAPH:
    evo_graph_cv, evo_graph_ext = graph.run_graph_grid(
        ctx.stores,
        ctx.fold_ids,
        BASE,
        EVO_GRAPH_EXPERIMENTS,
        run_version='ultimate_evo_graph_v1',
        cfg=graph.GraphConfig(),
        device=DEVICE,
    )
    display(evo_graph_cv)
    display(evo_graph_ext.sort_values(['dataset','AUPRC'], ascending=[True,False]))


# Part H — Frozen experiment history / sanity check

The table below is **reference only**. It is not substituted for rerunning an experiment. Its role is to catch major pipeline regressions: a fresh run should be in the same performance regime unless software, cache, split or protocol has deliberately changed.


In [ ]:
display(suite.known_result_history())


,representation,dataset,AUPRC,AUROC,F1@0.5,MCC@0.5
0,Historical KAN,Test300,0.3110,0.7965,0.3902,0.4048
1,Historical KAN,Test46,0.5634,0.9529,0.6713,0.6696
2,ESM+Atomic,Test300,0.3578,0.8275,0.3214,0.3256
3,ESM+Atomic,Test46,0.6520,0.9634,0.4552,0.4942
4,ESM+Chem,Test300,0.3525,0.8297,0.2963,0.3093
5,ESM+Chem,Test46,0.6784,0.9719,0.3743,0.4372
6,ESM+Atomic+Chem,Test300,0.3694,0.8342,0.2905,0.3045
7,ESM+Atomic+Chem,Test46,0.6655,0.9659,0.4172,0.4689
8,ESM+MULTI2,Test300,0.3718,0.8384,0.2987,0.3130
9,ESM+MULTI2,Test46,0.6641,0.9655,0.4013,0.4537


## Historically observed core results

### Standardized MLP Dev CV AUPRC

| Representation | Mean AUPRC |
|---|---:|
| AA | ~0.0763 |
| AA+Structure | ~0.2679 |
| AA+Atomic | ~0.3813 |
| AA+Chem | ~0.4056 |
| AA+Atomic+Chem | ~0.4106 |
| ESM | ~0.4023 |
| ESM+Structure | ~0.4049 |
| ESM+Atomic | ~0.4393 |
| ESM+Chem | ~0.4431 |
| ESM+Atomic+Chem | ~0.4473 |
| ESM+Atomic+Chem+MULTI2 | ~0.4584 in the merged mechanistic run |

### Final external AUPRC

| Representation | Test300 | Test46 |
|---|---:|---:|
| Historical ESM+Structure+KAN | 0.3110 | 0.5634 |
| ESM+Atomic | 0.3578 | 0.6520 |
| ESM+Chem | 0.3525 | 0.6784 |
| ESM+Atomic+Chem | 0.3694 | 0.6655 |
| ESM+MULTI2 | 0.3718 | 0.6641 |
| **ESM+Atomic+Chem+MULTI2** | **0.3842** | **0.6936** |

The final representation therefore improved external AUPRC by roughly **23% relative to the historical baseline on both external sets**, but it did not dominate every AUROC/F1/recall metric.


# Part I — Prediction overlap / residual-error diagnostics

AUPRC alone does not tell us whether a new representation merely rescales easy examples or actually corrects previously missed binding residues.

For a set of ensemble prediction CSVs, use the helpers below to calculate:

- TP/FN transitions across representations
- how many ESM false negatives are rescued
- how many ESM true positives are lost
- net positive-residue change

The historical observation was that the mechanism features modify a **small, hard subset** rather than replacing the ESM decision boundary wholesale. This is the motivation for the residual graph architecture.


In [ ]:
# Example only — run after you have ensemble prediction CSVs from a selected run.
#
# frames = {
#     'ESM': reporting.load_ensemble_csv(BASE, 'ultimate_final_focused_v1', 'MLP', 'ESM', 'TestSet300'),
#     'Atomic': reporting.load_ensemble_csv(BASE, 'ultimate_final_focused_v1', 'MLP', 'ESM+Atomic', 'TestSet300'),
#     'Final': reporting.load_ensemble_csv(BASE, 'ultimate_final_focused_v1', 'MLP', 'ESM+Atomic+Chem+MULTI2', 'TestSet300'),
# }
# display(reporting.positive_transition_table(frames, threshold=.5))
# display(pd.DataFrame([reporting.esm_origin_rescue(frames, esm_key='ESM', final_key='Final', threshold=.5)]))


# Part J — Scientific interpretation / next decision

Current evidence supports the following hierarchy:

### 1. Chemistry and coordination are real signal

AA alone is weak. Adding explicit donor chemistry and atomic coordination produces a very large gain, even without ESM. This argues that the features are not arbitrary model decorations; they encode task-relevant binding physics/chemistry.

### 2. Generic coarse structure is mostly redundant after ESM

SPATIAL/LOCAL help strongly without ESM but add little once ESM is present.

### 3. Atomic and graph relations overlap

The clean `AA+Chem` graph showed a modest gain over the independent-residue MLP, but adding ATOM to the node largely removed that advantage. This is consistent with ATOM acting as a handcrafted compression of local coordination relations.

### 4. More local descriptors are not automatically better

The 453-d maximal no-ESM graph was worse than the compact mechanism-first MLP. The design lesson is:

\[
\textbf{better relation/orthogonal information} > \textbf{more redundant local dimensions}
\]

### 5. The remaining ESM advantage is likely contextual

The leading hypotheses are:

- evolutionary conservation / functional prior
- whole-protein or domain context
- long-range sequence semantics not equivalent to local spatial neighbors

That is why **EVO48 is the next priority**. If EVO closes much of the Dev gap, follow with coevolution edges / global context. If EVO does not help, do not continue adding local geometric scalars blindly.


# Part K — Output locations

Each modern run writes to:

```text
/content/drive/MyDrive/Protein-binding-Phi/results/<run_version>/metal_binding/
```

Typical artifacts include:

```text
fold_predictions/
fold_metrics.csv
cv_summary.csv
external_summary.csv
paired_custom_AUPRC.csv        # where requested
```

Persistent feature caches remain under:

```text
/content/drive/MyDrive/Protein-binding-Phi/cache/
```

MSA/EVO artifacts are stored under:

```text
/content/drive/MyDrive/Protein-binding-Phi/cache/evolution_v1/
```

Because run versions are unique, this notebook is designed to **resume rather than overwrite** the historical standardized experiments.


In [ ]:
# Final environment snapshot for the run log.
import platform, json
snapshot = {
    'gear_ultimate_version': GEAR_ULTIMATE_VERSION,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'device': str(DEVICE),
    'base': str(BASE),
    'kan_available': bool(core.HAS_KAN),
}
print(json.dumps(snapshot, indent=2))


{
  "gear_ultimate_version": "1.0.0",
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "device": "cuda",
  "base": "/content/drive/MyDrive/Protein-binding-Phi",
  "kan_available": true
}
